Cambios: 
- Implementacion noiseless simple
- Simulación via qiskit_aer (en GPUs)
- Optimicación pytorch para evaluacion en multiples GPUs

## Implementation (statevector simulation)

In [99]:
#--- INSTALATION INSTRUCTIONS ---#

# For linux 64-bit systems,
#uname -a

# Conda quick installation
#mkdir -p ~/miniconda3
#wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O ~/miniconda3/miniconda.sh
#bash ~/miniconda3/miniconda.sh -b -u -p ~/miniconda3
#rm ~/miniconda3/miniconda.sh

# Create enviroment with conda
#conda create -n myenv python=3.10
#conda activate myenv
#pip install qiskit qiskit-machine-learning 'qiskit-machine-learning[sparse]' qiskit_aer qiskit-aer-gpu qiskit_algorithms torch matplotlib pylatexenc ipykernelc
# IMPORTANT: Make sure you are on 3.10
# May need to restart the kernel after instalation

#--- Imports ---#
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import random_statevector, Statevector, SparsePauliOp
from qiskit.circuit.library import real_amplitudes, efficient_su2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit import qpy

from qiskit_machine_learning.neural_networks import EstimatorQNN, SamplerQNN
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient, SPSAEstimatorGradient
from qiskit_machine_learning.connectors import TorchConnector

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import EstimatorV2 as EstimatorV2_sim, SamplerV2 as SamplerV2_sim
from qiskit_aer.quantum_info import AerStatevector
from qiskit_aer.library import SaveProbabilities

from qiskit_ibm_runtime import EstimatorV2 as EstimatorV2_rh, QiskitRuntimeService, Session
from qiskit_ibm_runtime.options import EstimatorOptions

from qiskit_algorithms.gradients import ReverseEstimatorGradient

import numpy as np
import torch
import matplotlib.pyplot as plt
import time
import os
import signal
import datetime as dt
import pickle

In [ ]:
#- Configuration -#

# Training configuration dict
train_config = {
    'execution_type': "noisy", # 'noiseless', 'noisy' or 'real'
    'n_qubits': 2,
    'seed': 4,
    'id': None, # For different circuits or training parameters
    
    # Data
    'reset_data': True,
    'create_circuits': True, # Create circuits or load from file

    # Training parameters
    'max_iterations': 1000,
    'gen_iterations': 1,
    'disc_iterations': 3,
    'save_loss_iterations': 1, # Calculate extra forward pass to save loss
    'print_progress_iterations': 1,

    # Compute methods
    'device': "CPU", # 'GPU' or 'CPU'
    'gradient_method': "SPSA", # qiskit_algorithms.gradients For now: 'PSR', 'SPSA' and 'REG'
    'TorchConnector': True, # Use TorchConnector QNN instances

    # File names
    'training_data_file': None, # If all None, automatically created with manage_files function
    'circuits_file': None,
    'backend_file': None 
}


# File management
def manage_files(
        data_folder_name = 'data', 
        implementation_name = 'noiseless_torch_opt', 
        execution_type_name = train_config['execution_type'], 
        training_data_file_name = 'training_data', 
        circuits_file_name = 'circuits', 
        backend_file_name = 'backend'
        ):
    
    data_folder = data_folder_name + '/' + implementation_name + '/' + execution_type_name + '/' + 'q' + str(train_config['n_qubits']) + '/' + 'seed' + str(train_config['seed']) + '/'
    if train_config['id'] is not None:
        data_folder = data_folder + '/' + str(train_config['id']) + '/' 
    training_data_file = data_folder + training_data_file_name + '.pth'
    circuits_file = data_folder + circuits_file_name + '.qpy'
    backend_file = data_folder + backend_file_name + '.pkl'

    # Create folders if they do not exist
    if not os.path.exists(data_folder):
        os.makedirs(data_folder)

    return training_data_file, circuits_file, backend_file



if ((train_config['training_data_file'] is None) and (train_config['circuits_file'] is None) and (train_config['backend_file'] is None)):
    train_config['training_data_file'], train_config['circuits_file'], train_config['backend_file'] = manage_files()

In [ ]:
#- Backend configuration -#

# Get shots from precision
def get_shots(precision):
    if np.isclose(precision, 0, atol=1e-8):
        return None
    else:
        return int(1/(precision**2))


# Backend configuration
backend_config = {
    # Real backend
    'name': "ibm_basquecountry",
    'channel': "ibm_quantum_platform",

    # Noisy backend
    'reset_backend': False, # Get current backend state or load from file
    'timestamp': dt.datetime.now(),

    # Noiseless/Noiseless backend
    'sim_options': {
        'method': 'automatic', # TODO If None, use defalut options (los de abajo)
        'precision': "double", # 'single' or 'double'. Also used to choose torch dtype
        'seed_simulator': train_config['seed']
    },
}

# Backend configuration for GPU cuda execution
if train_config["device"] == "GPU":
    backend_config['sim_options'].update({
        'device': 'GPU', # For nvidia cuda

        'cuStateVec_enable': True,   # NVIDIA library optimization
        'batched_shots_gpu': True,   # Parallelize batch on GPU
        'blocking_enable': False,     # Disable chunking; simulation fits in VRAM 
        #'target_gpus':[0,1], # TODO import torch after? In .py read how many after printing aviables?

        'runtime_parmeter_bind_enable': True, # tells Aer to keep the circuit parameterized and bind the numeric values at execution time TODO prueba pa saber las mejores options
    })

# Backend configuration for noise
if train_config["execution_type"] in ["noisy", "real"]:
    backend_config['sim_options']['method'] = 'density_matrix'
    backend_config['train_precision'] = 0.015625
else:
    backend_config['sim_options']['method'] = 'statevector'
    backend_config['train_precision'] = 0.0
backend_config['sim_options']['shots'] = get_shots(backend_config["train_precision"])
backend_config['eval_precision'] = 0.015625 # Only used with TorchConnector: precision cannot be 0 because of Aer SamplerV2
'''
method:
    For noiseless execution:
        - statevector
        - matrix_product_state: more qubits, low entanglement
    For noisy execution:
        - density_matrix
        - statevector: uses less memory
    - stabilizer: only for Clifford circuits (only H, CNOT, and S gates)
    - tensor_network: for large circuits (when reaching memory limits, only GPU)
'''

'''
# More options
backend_for_info = AerSimulator()
print("AerSimulator backend configuration options:")
for option in backend_for_info.options:
    print(" -", option)

'''



# # Save account
# QiskitRuntimeService.save_account(
#     token="",
#     instance="crn:v1:bluemix:public:quantum-computing:eu-de:a/cb804b30dfcb48b890393bfd6e41e9c2:4cb40c64-a531-4c13-b39c-e04c31185259::",
#     set_as_default = True,
#     overwrite=True
# )



# Save current backend
def reset_backend():
    if train_config['execution_type'] in ["noisy", "real"]:
        service = QiskitRuntimeService(channel=backend_config['channel']) # Real backend    #job.properties().to_dict() to save real hardware state during training TODO
        real_backend = service.backend(backend_config['name']) #backend = service.least_busy(min_num_qubits=30)
        backend = AerSimulator.from_backend(real_backend, **backend_config['sim_options']) # Get current backend state
    else:
        backend = AerSimulator(**backend_config['sim_options'])

    backend_dict = {
        'timestamp': dt.datetime.now(dt.timezone.utc),
        'configuration': backend.configuration(),
        'properties': backend.properties(),
        'target': backend.target,
        'options': backend.options,
        'noise_model': None if train_config["execution_type"] == "noiseless" else backend.options.noise_model
    }

    with open(train_config['backend_file'], "wb") as f:
        pickle.dump(backend_dict, f)


# Reset backend file
if backend_config['reset_backend']:
    reset_backend()


# Create backend
if train_config['execution_type'] == "real": # TODO guardar otras configuraciones del backend real?
    service = QiskitRuntimeService(channel=backend_config['channel']) # Real backend
    real_backend = service.backend(backend_config['name']) #backend = service.least_busy(min_num_qubits=30)
    backend = real_backend
    session = Session(backend=backend)
    # TODO qiskit-ibm-runtime

else:
    # Load backend configuration
    try:
        with open(train_config['backend_file'], "rb") as f:
            backend_dict = pickle.load(f)
    except FileNotFoundError:
        print("Backend data file not found. Resetting backend configuration.")
        reset_backend()
        with open(train_config['backend_file'], "rb") as f:
            backend_dict = pickle.load(f)


    # Create backend
    backend = AerSimulator(
        configuration=backend_dict['configuration'],
        properties=backend_dict['properties'],
        target=backend_dict['target'],
        **backend_dict['options']
    )


    # Create Estimator for simulation
    estimator = EstimatorV2_sim(
        options = {
            "default_precision": backend_config["train_precision"],
            "backend_options": backend.options,
        })


# Backend for noiseless evaluation
eval_backend = AerSimulator(**backend_config['sim_options']) # Noisy eval? TODO por ahora no


# Transpilation method
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)


# Select device torch
#os.environ["CUDA_VISIBLE_DEVICES"] = "1,2" # before torch import to select specific devices
#import torch
if train_config['device'] == "GPU" and torch.cuda.is_available():
    print(f"GPUs available to PyTorch: {torch.cuda.device_count()}")
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

if backend_config["sim_options"]["precision"] == "double":
    dtype = torch.float64
else:
    dtype = torch.float32


# Print backend properties
print(backend)

AerSimulator('aer_simulator_from(ibm_basquecountry)'
             noise_model=<NoiseModel on ['sx', 'x', 'measure', 'id', 'reset', 'cz']>)


In [102]:
#- Create quantum circuits -#

# Create real data sample circuit
def generate_real_circuit():
    n_qubits = train_config['n_qubits']

    # sv = random_statevector(2**n_qubits, seed=train_config['seed'])
    # qc = QuantumCircuit(n_qubits)
    # qc.prepare_state(sv, qc.qubits, normalize=True)

    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits-1))
    qc.cx(n_qubits-2, n_qubits-1)
    return qc


# Create generator
def generate_generator():
    n_qubits = train_config['n_qubits']

    qc = real_amplitudes(n_qubits,
                        reps=3, # Number of layers
                        parameter_prefix='θ_g',
                        name='Generator')
    
    return qc.decompose()


# Create discriminator
def generate_discriminator():
    n_qubits = train_config['n_qubits']

    qc = efficient_su2(n_qubits,
                      entanglement="reverse_linear",
                      reps=1, # Number of layers
                      parameter_prefix='θ_d',
                      name='Discriminator').decompose()


    param_index = qc.num_parameters

    for i in reversed(range(n_qubits - 1)):
        qc.cx(i, n_qubits - 1)

    #qc.rx(disc_weights[param_index], N_QUBITS-1); param_index += 1
    qc.ry(Parameter("θ_d["+str(param_index)+"]"), n_qubits-1); param_index += 1
    qc.rz(Parameter("θ_d["+str(param_index)+"]"), n_qubits-1); param_index += 1
    
    return qc


# Create quantum circuits
def create_circuits():
    real_circuit = generate_real_circuit()
    generator_circuit = generate_generator()
    discriminator_circuit = generate_discriminator()

    # Save circuits in a file
    with open(train_config['circuits_file'], 'wb') as fd:
        qpy.dump([real_circuit, generator_circuit, discriminator_circuit], fd)


# Rewrite circuits file if indicated in options
if train_config['create_circuits']:
    create_circuits()


# Load circuits from file
try:
    with open(train_config['circuits_file'], 'rb') as fd:
        circuits = qpy.load(fd)
except FileNotFoundError:
    print("Circuits file not found. Creating new circuits file.")
    create_circuits()
    with open(train_config['circuits_file'], 'rb') as fd:
        circuits = qpy.load(fd)
    
    
real_circuit = circuits[0]
generator_circuit = circuits[1]
discriminator_circuit = circuits[2]

In [103]:
#- Set up training quantum circuits -#
def generate_training_circuits(real_circuit, generator_circuit, discriminator_circuit):
    n_qubits = train_config['n_qubits']

    # Connect real data and discriminator
    real_disc_circuit = QuantumCircuit(n_qubits)
    real_disc_circuit.compose(real_circuit, inplace=True)
    real_disc_circuit.compose(discriminator_circuit, inplace=True)

    # Connect generator and discriminator
    gen_disc_circuit = QuantumCircuit(n_qubits)
    gen_disc_circuit.compose(generator_circuit, inplace=True)
    gen_disc_circuit.compose(discriminator_circuit, inplace=True)


    # Gradient computation method
    if train_config['gradient_method'] == 'SPSA':
        gradient = SPSAEstimatorGradient(estimator=estimator)
    elif train_config['gradient_method'] == 'REG':
        gradient = ReverseEstimatorGradient()
    else:
        gradient = ParamShiftEstimatorGradient(estimator=estimator)


    # Observables
    H1 = SparsePauliOp.from_list([("Z" + "I"*(n_qubits-1), 1.0)])
    #obs_gen_eval = [SparsePauliOp.from_list([("I" * i + "Z" + "I" * (n_qubits - 1 - i), 1)]) for i in range(n_qubits)] TODO pa angle


    # Transpilation
    real_disc_circuit_transpiled, gen_disc_circuit_tranpiled = pm.run([real_disc_circuit, gen_disc_circuit])
    obs_real_disc = [H1.apply_layout(real_disc_circuit_transpiled.layout)]
    obs_gen_disc = [H1.apply_layout(gen_disc_circuit_tranpiled.layout)]


    N_DPARAMS = discriminator_circuit.num_parameters

    # specify QNN to update generator parameters
    gen_qnn = EstimatorQNN(circuit=gen_disc_circuit_tranpiled,
                        input_params=gen_disc_circuit_tranpiled.parameters[:N_DPARAMS], # fixed parameters (discriminator parameters)
                        weight_params=gen_disc_circuit_tranpiled.parameters[N_DPARAMS:], # parameters to update (generator parameters)
                        estimator=estimator,
                        observables=obs_gen_disc,
                        gradient=gradient,
                        default_precision=backend_config["train_precision"],
                        #pass_manager=pm,
                        input_gradients=train_config['TorchConnector']
                        )

    # specify QNN to update discriminator parameters regarding to fake data
    disc_fake_qnn = EstimatorQNN(circuit=gen_disc_circuit_tranpiled,
                            input_params=gen_disc_circuit_tranpiled.parameters[N_DPARAMS:], # fixed parameters (generator parameters)
                            weight_params=gen_disc_circuit_tranpiled.parameters[:N_DPARAMS], # parameters to update (discriminator parameters)
                            estimator=estimator,
                            observables=obs_gen_disc,
                            gradient=gradient,
                            default_precision=backend_config["train_precision"],
                            pass_manager=pm,
                            input_gradients=train_config['TorchConnector']
                            )

    # specify QNN to update discriminator parameters regarding to real data
    disc_real_qnn = EstimatorQNN(circuit=real_disc_circuit_transpiled,
                            input_params=[], # no input parameters
                            weight_params=real_disc_circuit_transpiled.parameters[:N_DPARAMS], # parameters to update (discriminator parameters)
                            estimator=estimator,
                            observables=obs_real_disc,
                            gradient=gradient,
                            default_precision=backend_config["train_precision"],
                            pass_manager=pm,
                            input_gradients=train_config['TorchConnector']
                            )
    
    # specify Generator evaluator
    gen_eval_circuit = generator_circuit.copy()
    if train_config['TorchConnector']:
        # Create Sampler for evaluation, to use with TorchConnector
        sampler = SamplerV2_sim(
            default_shots = get_shots(backend_config['eval_precision']),
            options = {"backend_options": eval_backend.options}
        )
        # # Noiseless sampler (not BaseV2, cannot be used for TorchConnector)
        # sampler = Sampler_sim(
        #     backend_options = backend.options,
        #     run_options = {
        #         'shots': None,
        #     },
        #     skip_transpilation=True
        # )

        # specify QNN to evaluate generator
        gen_eval_circuit.measure_all()
        gen_eval_transpiled = SamplerQNN(circuit=gen_eval_circuit,
                                input_params=[], # no input parameters
                                weight_params=gen_eval_circuit.parameters,
                                sampler=sampler,
                                gradient=gradient,
                                #pass_manager=pm, # eval_pm? TODO
                                input_gradients=train_config['TorchConnector']
                                )
        # gen_eval_transpiled = EstimatorQNN(circuit=gen_eval_circuit, TODO pa angle
        #                         input_params=[], # no input parameters
        #                         weight_params=gen_eval_circuit.parameters,
        #                         estimator=estimator,
        #                         observables=obs_gen_eval,
        #                         gradient=gradient,
        #                         default_precision=backend_config["train_precision"],
        #                         pass_manager=pm,
        #                         input_gradients=train_config['TorchConnector']
        #                         )
    else:
        gen_eval_circuit.append(SaveProbabilities(gen_eval_circuit.num_qubits), gen_eval_circuit.qubits)
        gen_eval_transpiled = gen_eval_circuit # Crear pm_eval? TODO
    

    return gen_qnn, disc_fake_qnn, disc_real_qnn, gen_eval_transpiled

gen_qnn, disc_fake_qnn, disc_real_qnn, gen_eval_transpiled = generate_training_circuits(real_circuit, generator_circuit, discriminator_circuit)

In [104]:
if train_config['TorchConnector']: 
    #f_loss = torch.nn.MSELoss(reduction="sum")
    class FLoss(torch.nn.Module):
        def __init__(self, reduction='sum'):
            super(FLoss, self).__init__()
            self.reduction = reduction

        def forward(self, x, label):
            loss = x * label
            
            if self.reduction == 'mean': # Para batches TODO
                return loss.mean()
            elif self.reduction == 'sum':
                return loss.sum()
            return loss
        
    f_loss = FLoss()

In [105]:
#- Restore parameters and model states -#

# Reset all data training
def reset_data(n_gen_params, n_disc_params):
    np.random.seed(train_config['seed'])

    init_gen_params = np.random.uniform(low=-np.pi, high=np.pi, size=(n_gen_params,)) * 0.1 # Start from near 0 parameters to mitigate drastic changes at the start
    init_disc_params = np.random.uniform(low=-np.pi, high=np.pi, size=(n_disc_params,)) * 0.1

    gen_params = torch.tensor(init_gen_params, requires_grad=True, dtype=dtype)
    disc_params = torch.tensor(init_disc_params, requires_grad=True, dtype=dtype)

    params = {
        'init_gen_params': init_gen_params,
        'init_disc_params': init_disc_params,
        'gen_params': gen_params,
        'disc_params': disc_params,
        'best_gen_params': init_gen_params,
        'current_epoch': 0,
        "metrics": {
            "gloss": {},
            "dloss": {},
            "eval": {},
            'times': {},
        },
        'random_state': np.random.get_state()
    }

    if train_config['TorchConnector']:
        model_g = TorchConnector(gen_qnn, initial_weights=init_gen_params)
        model_dr = TorchConnector(disc_real_qnn, initial_weights=init_disc_params)
        model_df = TorchConnector(disc_fake_qnn)
        eval_g = TorchConnector(gen_eval_transpiled)

        # Force model2 to look at model1's weights
        if 'weight' in model_df._parameters:
            model_df._parameters.pop('weight')
        if '_weights' in model_df._parameters:
            model_df._parameters.pop('_weights')
        model_df._parameters['weight'] = model_dr.weight
        model_df._parameters['_weights'] = model_dr.weight

        if 'weight' in eval_g._parameters:
            eval_g._parameters.pop('weight')
        if '_weights' in eval_g._parameters:
            eval_g._parameters.pop('_weights')
        eval_g._parameters['weight'] = model_g.weight
        eval_g._parameters['_weights'] = model_g.weight

        model_g.train() 
        model_dr.train()
        model_df.train()
        eval_g.eval()

        optimizer_g = torch.optim.Adam(model_g.parameters(), lr=0.005)
        optimizer_d = torch.optim.Adam(model_dr.parameters(), lr=0.005)

        params['model_g_state'] = model_g.state_dict()
        params['model_dr_state'] = model_dr.state_dict()
        params['model_df_state'] = model_df.state_dict()
        params['eval_g_state'] = eval_g.state_dict()
    
    else:
        optimizer_g = torch.optim.Adam([gen_params], lr=0.005)
        optimizer_d = torch.optim.Adam([disc_params], lr=0.005)

    params['optimizer_g_state'] = optimizer_g.state_dict()
    params['optimizer_d_state'] = optimizer_d.state_dict()

    torch.save(params, train_config['training_data_file'])


# Load parameters and training states
if train_config['reset_data']:
    reset_data(generator_circuit.num_parameters, discriminator_circuit.num_parameters)

try:
    params = torch.load(train_config['training_data_file'], weights_only=False)
except FileNotFoundError:
    print("Training data file not found. Resetting parameters.")
    reset_data(generator_circuit.num_parameters, discriminator_circuit.num_parameters)
    params = torch.load(train_config['training_data_file'], weights_only=False)

np.random.set_state(params['random_state'])

gen_params = params['gen_params']
disc_params = params['disc_params']

current_epoch = params['current_epoch']
gloss = params['metrics']['gloss']
gen_loss = list(gloss)[-1] if (gloss) else None
dloss = params['metrics']['dloss']
disc_loss = list(dloss)[-1] if (dloss) else None
eval = params['metrics']['eval']
min_eval = np.min(list(eval.values())) if (eval) else float('inf')
best_gen_params = params['best_gen_params']
times = params['metrics']['times']


if train_config['TorchConnector']:
    model_g = TorchConnector(gen_qnn)    
    model_dr = TorchConnector(disc_real_qnn)
    model_df = TorchConnector(disc_fake_qnn)
    eval_g = TorchConnector(gen_eval_transpiled)
    
    model_g.load_state_dict(params['model_g_state'])
    model_dr.load_state_dict(params['model_dr_state'])
    model_df.load_state_dict(params['model_df_state'])
    eval_g.load_state_dict(params['eval_g_state'])

    model_g.to(device)
    model_dr.to(device)
    model_df.to(device)
    eval_g.to(device)

    # Force model2 to look at model1's weights
    if 'weight' in model_df._parameters:
        model_df._parameters.pop('weight')
    if '_weights' in model_df._parameters:
        model_df._parameters.pop('_weights')
    model_df._parameters['weight'] = model_dr.weight
    model_df._parameters['_weights'] = model_dr.weight

    if 'weight' in eval_g._parameters:
        eval_g._parameters.pop('weight')
    if '_weights' in eval_g._parameters:
        eval_g._parameters.pop('_weights')
    eval_g._parameters['weight'] = model_g.weight
    eval_g._parameters['_weights'] = model_g.weight

    optimizer_g = torch.optim.Adam(model_g.parameters())
    optimizer_d = torch.optim.Adam(model_dr.parameters())

else:
    optimizer_g = torch.optim.Adam([gen_params])
    optimizer_d = torch.optim.Adam([disc_params])

optimizer_g.load_state_dict(params['optimizer_g_state'])
optimizer_d.load_state_dict(params['optimizer_d_state'])

In [106]:
#- Manage training interruption -#

# Class to manage training interruption
class Interrupter:
    def __init__(self):
        self.kill_now = False
        self.interrupt_count = 0

        # Intercept the Ctrl+C signal
        signal.signal(signal.SIGINT, self.handle_signal)
        # Intercept the termination signal (useful for Docker/systems)
        #signal.signal(signal.SIGTERM, self.handle_signal)

    def handle_signal(self, signum, frame):
        self.interrupt_count += 1
        
        if self.interrupt_count == 1:
            # First Press: Enable graceful exit
            self.kill_now = True
            print("\nInterrupter: Termination signal received. The loop will stop after the current iteration. (Press Ctrl+C again to force quit)")
        
        elif self.interrupt_count >= 2:
            # Second Press: Force quit immediately
            print("\nInterrupter: [!] Force quit triggered! Terminating immediately.")
            # Restore default signal handler to avoid recursion
            signal.signal(signal.SIGINT, signal.SIG_DFL)
            # Raise the exception to stop execution right here
            raise KeyboardInterrupt

In [107]:
#- Evualuation method -#

# Evaluation method: KL-Div of generated (ger_dist) and real (target) sample
def evaluate(gen_dist, target):
    return torch.nn.functional.kl_div(
        input = gen_dist.clamp_min(1e-10).log(), #torch.nn.functional.log_softmax(gen_dist, dim=-1),
        target = target, 
        reduction = 'sum' 
    ).item()

In [108]:
#- Forward and backward pass -#

if train_config['TorchConnector']:
    real_label = torch.ones(1, dtype=dtype, device=device)
    fake_label = torch.tensor(-1, dtype=dtype, device=device)

    # Discriminator pass
    def disc_pass(compute_forward):
        optimizer_d.zero_grad()

        # Calculate discriminator gradient with real data
        real_output = model_dr()
        real_loss = f_loss(real_output, real_label) # 1-> Real guess (correct)
        real_loss.backward(retain_graph=True)

        # Calculate discriminator gradient with generated data
        gen_params = torch.nn.utils.parameters_to_vector(model_g.parameters()).detach() #gen_params = optimizer_g.param_groups[0]['params'][0].detach()
        fake_output = model_df(gen_params)
        fake_loss = f_loss(fake_output, fake_label) # -1-> Fake guess (correct)
        fake_loss.backward()

        optimizer_d.step()

        # Calculate discriminator cost
        if (compute_forward):
            disc_loss = (real_loss.item() + fake_loss.item() -2)/4
        else:
            disc_loss = None

        return disc_loss

    # Generator pass
    def gen_pass(compute_forward):
        optimizer_g.zero_grad()

        # Calculate generator gradient
        disc_params = torch.nn.utils.parameters_to_vector(model_dr.parameters()).detach() #disc_params = optimizer_d.param_groups[0]['params'][0].detach()
        gen_output = model_g(disc_params)
        gen_loss = f_loss(gen_output, real_label) # 1-> Real guess (decieved)
        gen_loss.backward()  # Backward pass

        optimizer_g.step()

        # Calculate generator cost
        if compute_forward:
            gen_loss = (gen_loss.item() -1)/2
        else:
            gen_loss = None
        
        return gen_loss

    # Evaluate performance
    def evaluation(real_distribution_tensor):
        gen_dist = eval_g()

        # Performance measurement function: uses Kullback Leibler Divergence to measures the distance between two distributions
        current_kl = evaluate(gen_dist, real_distribution_tensor)

        return current_kl
    
    # Copy parameters
    def copy_params():
        torch.nn.utils.parameters_to_vector(model_g.parameters()).detach().numpy().copy()

else:
    gen_params_np = gen_params.detach().cpu().numpy()
    disc_params_np = disc_params.detach().cpu().numpy()

    # Discriminator pass
    def disc_pass(compute_forward):
        # Calculate discriminator cost
        if compute_forward:
            value_dcost_fake = disc_fake_qnn.forward(gen_params_np, disc_params_np)[0,0]
            value_dcost_real = disc_real_qnn.forward([], disc_params_np)[0,0]
            disc_loss = ((value_dcost_real - value_dcost_fake)-2)/4
        else:
            disc_loss = None

        # Caltulate discriminator gradient
        grad_dcost_fake = disc_fake_qnn.backward(gen_params_np, disc_params_np)[1][0,0]
        grad_dcost_real = disc_real_qnn.backward([], disc_params_np)[1][0,0]
        grad_dcost = grad_dcost_real - grad_dcost_fake
        
        # Update discriminator parameters
        optimizer_d.zero_grad()
        disc_params.grad = torch.tensor(grad_dcost, dtype=dtype)
        optimizer_d.step()

        return disc_loss

    # Generator pass
    def gen_pass(compute_forward):
        # Calculate generator cost
        if compute_forward:
            value_gcost = gen_qnn.forward(disc_params_np, gen_params_np)[0,0]
            gen_loss = (value_gcost-1)/2
        else:
            gen_loss = None

        # Calculate generator gradient
        grad_gcost = gen_qnn.backward(disc_params_np, gen_params_np)[1][0,0]

        # Update generator parameters
        optimizer_g.zero_grad()
        gen_params.grad = torch.tensor(grad_gcost, dtype=dtype)
        optimizer_g.step()

        return gen_loss

    # Evaluate performance
    def evaluation(real_distribution_tensor):
        # Get fake samples
        parameter_binds = {gen_eval_transpiled.parameters[i]: [gen_params_np[i]] for i in range(len(gen_eval_transpiled.parameters))}
        job = eval_backend.run([gen_eval_transpiled], parameter_binds=[parameter_binds])
        result = job.result()
        all_counts = result.data(0)['probabilities']
        gen_dist = torch.tensor(all_counts, dtype=dtype, device=device)

        # Performance measurement function: uses Kullback Leibler Divergence to measures the distance between two distributions
        current_kl = evaluate(gen_dist, real_distribution_tensor)

        return current_kl
    
    # Copy parameters
    def copy_params():
        return gen_params_np.copy()


In [109]:
#- Training -#

D_STEPS = train_config['disc_iterations']
G_STEPS = train_config['gen_iterations']
C_STEPS = train_config['save_loss_iterations']

real_distribution_tensor = torch.tensor(Statevector(real_circuit).probabilities(), dtype=dtype, device=device) # Retrieve real data probability distribution 

interrupter = Interrupter()

if train_config['print_progress_iterations']:
    TABLE_HEADERS = "Epoch | Generator cost | Discriminator cost | Eval | Best eval | Time |"
    print(TABLE_HEADERS)

prev_times = 0
start_time = time.time()

#--- Training loop ---#
try: # In case of interruption
    for epoch in range(current_epoch, train_config['max_iterations']+1):

        #--- Quantum discriminator parameter updates ---#
        for disc_train_step in range(D_STEPS):
            cost = disc_pass((disc_train_step == D_STEPS - 1) and (epoch % C_STEPS == 0))
            if cost is not None: 
                disc_loss = cost
                dloss[epoch] = disc_loss


        #--- Quantum generator parameter updates ---#
        for gen_train_step in range(G_STEPS):
            cost = gen_pass((gen_train_step == G_STEPS - 1) and (epoch % C_STEPS == 0))
            if cost is not None: 
                gen_loss = cost
                gloss[epoch] = gen_loss


        #--- Track KL and save best performing generator weights ---#
        current_kl = evaluation(real_distribution_tensor)
        eval[epoch] = current_kl
        if min_eval > current_kl:
            min_eval = current_kl
            best_gen_params = copy_params() # New best
        

        # Calculate time
        cur_time = (time.time() - start_time)
        times[epoch] = cur_time
        start_time = time.time()


        #--- Print progress ---#
        if train_config['print_progress_iterations'] and (epoch % train_config['print_progress_iterations'] == 0):
            now_times = sum(times.values())
            for header, val in zip(TABLE_HEADERS.split('|'),
                                (epoch, gen_loss, disc_loss, current_kl, min_eval, now_times - prev_times)):
                print(f"{val:.3g} ".rjust(len(header)), end="|")
            print()

            prev_times = now_times


        # In case of interruption
        if interrupter.kill_now:
            print("Interrupter: Graceful exit triggered. Breaking loop.")
            break
            
#--- Save parameters and optimizer states data ---#
finally:
    params = {
        'init_gen_params': params['init_gen_params'],
        'init_disc_params': params['init_disc_params'],
        'best_gen_params': best_gen_params,
        'gen_params': gen_params,
        'disc_params': disc_params,
        'current_epoch': epoch+1,
        "metrics": {
            "gloss": gloss,
            "dloss": dloss,
            "eval": eval,
            'times': times,
        },
        'optimizer_g_state': optimizer_g.state_dict(),
        'optimizer_d_state': optimizer_d.state_dict(),
    }

    if train_config['TorchConnector']:
        params['model_g_state'] = model_g.state_dict()
        params['model_dr_state'] = model_dr.state_dict()
        params['model_df_state'] = model_df.state_dict()
        params['eval_g_state'] = eval_g.state_dict()
    
    torch.save(params, train_config['training_data_file'])
    
    eval_data = list(eval.values()) if eval else [0]
    print("Training complete:", "\n   Data path:", train_config['training_data_file'], "\n   Best eval:", np.min(eval_data), "in epoch", np.argmin(eval_data), "\n   Improvement:", eval_data[0]-np.min(eval_data), "\n   Total time:", sum(times.values()))

Epoch | Generator cost | Discriminator cost | Eval | Best eval | Time |
    0 |        -0.0565 |              -0.67 | 3.48 |      3.48 | 63.9 |
    1 |        -0.0429 |             -0.661 | 3.48 |      3.48 | 60.2 |
    2 |        -0.0448 |             -0.668 | 3.48 |      3.48 | 60.2 |
    3 |        -0.0401 |             -0.665 | 3.48 |      3.48 | 60.2 |

Interrupter: Termination signal received. The loop will stop after the current iteration. (Press Ctrl+C again to force quit)
    4 |        -0.0352 |             -0.662 | 3.48 |      3.48 | 62.8 |
Interrupter: Graceful exit triggered. Breaking loop.
Training complete: 
   Data path: data/noiseless_torch_opt/noisy/q2/seed2/training_data.pth 
   Best eval: 3.482114549768256 in epoch 4 
   Improvement: 0.0011365078389644623 
   Total time: 307.38386130332947
